In [18]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats
import statsmodels.api as sm
from sklearn.preprocessing import (
    PowerTransformer,
    OneHotEncoder,
    StandardScaler
)


ModuleNotFoundError: No module named 'statsmodels'

In [16]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from wordcloud import WordCloud, STOPWORDS
from IPython.display import display, HTML

ModuleNotFoundError: No module named 'wordcloud'

### Pair plots

In [4]:
def display_html(size=3, content="content"):
  display(HTML(f"<h{size}>{content}</h{size}>"))

In [5]:
def pair_plots(data,
               height = 3,
               aspect = 1.5,
               hue = None,
               legend = False):
    display_html(2, "Pair plots")
    pair_grid = sns.PairGrid(
    data = data,
    height = height,
    aspect = aspect,
    hue = hue,
    corner = True)
    pair_grid.map_lower(sns.scatterplot)

    if legend:
        pair_grid.add_legend()
    

### correlation plot function

In [1]:
def rotate_xlabels(ax, angle=35):
  ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=angle,
        ha="right"
    )

In [2]:
def rotate_ylabels(ax, angle=0):
  ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=angle
  )

In [3]:
def correlation_heatmap(data,
                        figsize = (12,6),
                        method = "spearman",
                        cmap = "RdBu"):
      cm = data.corr(method = method, numeric_only = True)
      mask = np.zeros_like(cm, dtype=bool)
      mask[np.triu_indices_from(mask)] = True
    
      fig, ax = plt.subplots(figsize=figsize)
      hm = sns.heatmap(
          cm,
          vmin=-1,
          vmax=1,
          cmap=cmap,
          center=0,
          annot=True,
          fmt=".2f",
          linewidths=1.5,
          square=True,
          mask=mask,
          ax=ax
      )
      rotate_xlabels(ax)
      rotate_ylabels(ax)
      ax.set(title=f"{method.title()} Correlation Matrix Heatmap")


### Cramers - v correlation heatmap

In [ ]:
def cramers_v(data, var1, var2):
    ct = pd.crosstab(data.loc[:, var1],
                     data.loc[:, var2]
                    )
    r,c = ct.shape
    n = ct.sum().sum()
    chi2 = stats.chi2_contingency(ct).statistic
    phi2 = chi2/n

    # bias correction
    phi2 = max(0, phi2 - ((r-1)*(c-1)/(n-1)))
    r_ = r - (((r-1) **2) / (n-1))
    c_ = c - (((c-1) ** 2) / (n-1))

    return np.sqrt(phi2_/min(r_-1, c_-1))

### gives detailed summary of numeric features

In [10]:
def num_summary(data, var):
    import warnings
    warnings.filterwarnings("ignore")

    # title
    col = data.loc[:, var].copy()
    display_html(size=2, content=var)

    # quick glance
    display_html(3, "Quick Glance:")
    display(col)

    # meta-data
    display_html(3, "Meta-data:")
    print(f"{'Data Type':15}: {col.dtype}")
    print(f"{'Missing Data':15}: {col.isna().sum():,} rows ({col.isna().mean() * 100:.2f} %)")
    print(f"{'Available Data':15}: {col.count():,} / {len(col):,} rows")

In [12]:
# quantiles
display_html(3, "Percentiles:")
display(
      col
      .quantile([0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0])
      .rename(index=lambda val: f"{val * 100:.0f}")
      .rename("value")
      .rename_axis(index="percentile")
      .to_frame()
)

NameError: name 'HTML' is not defined

In [ ]:
# central tendancy
  display_html(3, "Central Tendancy:")
  display(
      pd
      .Series({"mean": col.mean(),
               "trimmed mean (5%)": stats.trim_mean(col.values, 0.05),
               "trimmed mean (10%)": stats.trim_mean(col.values, 0.1),
               "median": col.median()})
      .rename("value")
      .to_frame()
  )